In [1]:
import sys 
import os 

sys.path.append(os.path.join('..', '..'))

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.base import clone
import gc
import pandas as pd
import numpy as np
import joblib
import json
from sklearn.inspection import permutation_importance

from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

In [3]:
data_dir = Path("../../data/base_de_dados_final/")
arquivo = data_dir / "base_feature_engineering.csv"

In [4]:
df_ml = pd.read_csv(arquivo, sep=";", encoding="latin1", dtype=str)

In [5]:
df_ml.head()

,ATENDIMENTO_BAIRRO_NOME,FLAG_EQUIPAMENTO_URBANO,FLAG_FLAGRANTE,NATUREZA1_DEFESA_CIVIL,NATUREZA2_DEFESA_CIVIL,NATUREZA3_DEFESA_CIVIL,NATUREZA4_DEFESA_CIVIL,NATUREZA5_DEFESA_CIVIL,OCORRENCIA_DIA_SEMANA,OCORRENCIA_MES,...,OUTROS,log_rendimento,log_pop,rendimento_medio_responsavel_sm_estimado_norm,pct_alfabetizacao_15mais_estimado_norm,pct_sem_banheiro_sanitario_estimado_norm,pct_esgotamento_precario_estimado_norm,pct_sem_rede_geral_agua_estimado_norm,pct_lixo_destino_inadequado_estimado_norm,IQV_final
0,cidade industrial,0,0,0,0,0,0,0,5,1,...,0,1.2725655957915476,12.054569291774312,0.10430247718383312,0.5860284605433375,0.8524590163934427,0.9580992133568791,0.9939548202354438,0.9832635983263598,54.575836660173394
1,fazendinha,1,0,0,0,0,0,0,5,1,...,0,1.4445632692438664,10.249025712063943,0.1479791395045633,0.693402328589909,0.9344262295081966,0.981377428158613,0.9977728285077951,0.9916317991631799,60.589451314297285
2,uberaba,0,0,0,0,0,0,0,5,1,...,1,1.5933085305042167,11.181751169442313,0.19230769230769235,0.693402328589909,0.819672131147541,0.9049606678439557,0.960229080496341,0.9832635983263598,60.09137967837168
3,sitio cercado,0,0,0,0,0,0,0,5,1,...,1,1.2412685890696329,11.664899491892095,0.0971316818774446,0.5795601552393275,0.8360655737704918,0.9849092952319795,0.9910913140311803,0.99581589958159,54.28874525901941
4,tatuquara,1,0,0,0,0,0,0,5,1,...,0,1.1085626195212777,10.856862100640145,0.06910039113428944,0.3984476067270371,0.721311475409836,0.8515010435061807,0.9882278078269169,0.9163179916317991,44.56291924850032


In [6]:
df_ml.columns

Index(['ATENDIMENTO_BAIRRO_NOME', 'FLAG_EQUIPAMENTO_URBANO', 'FLAG_FLAGRANTE',
       'NATUREZA1_DEFESA_CIVIL', 'NATUREZA2_DEFESA_CIVIL',
       'NATUREZA3_DEFESA_CIVIL', 'NATUREZA4_DEFESA_CIVIL',
       'NATUREZA5_DEFESA_CIVIL', 'OCORRENCIA_DIA_SEMANA', 'OCORRENCIA_MES',
       'OCORRENCIA_DIA', 'MADRUGADA', 'MANHA', 'TARDE', 'NOITE',
       'CRIME_VIOLENTO', 'ATENDIMENTO_OPERACIONAL_ASSISTENCIAL',
       'ACIDENTE_TRANSITO', 'CRIME_PATRIMONIAL', 'CRIME_DROGAS_SUBSTANCIAS',
       'CRIME_ORDEM_PUBLICA', 'ano', 'populacao_estimado', 'OUTROS',
       'log_rendimento', 'log_pop',
       'rendimento_medio_responsavel_sm_estimado_norm',
       'pct_alfabetizacao_15mais_estimado_norm',
       'pct_sem_banheiro_sanitario_estimado_norm',
       'pct_esgotamento_precario_estimado_norm',
       'pct_sem_rede_geral_agua_estimado_norm',
       'pct_lixo_destino_inadequado_estimado_norm', 'IQV_final'],
      dtype='str')

### Tipagem correta das variáveis

Antes de executar o pipeline de machine learning, é necessário garantir que todas as variáveis estejam com a tipagem adequada. No dataset utilizado, as variáveis forma organizadas em três tipos:

1. String: 
   - `ATENDIMENTO_BAIRRO_NOME`
  
2. Int:
    - `FLAG_EQUIPAMENTO_URBANO`;
    - `FLAG_FLAGRANTE`;
    - `OCORRENCIA_MES`;
    - `OCORRENCIA_DIA`;
    - `MADRUGADA`;
    - `MANHA`;
    - `TARDE`;
    - `NOITE`;
    - `CRIME_VIOLENTO`;
    - `ATENDIMENTO_OPERACIONAL_ASSISTENCIAL`;
    - `ACIDENTE_TRANSITO`;
    - `CRIME_PATRIMONIAL`;
    - `CRIME_DROGAS_SUBSTANCIAS`;
    - `CRIME_ORDEM_PUBLICA`;
    - `OUTROS`;
    - `ano`;
    - `populacao_estimado`.

3. Float:
   - `log_rendimento`;
   - `log_pop`;
   - `rendimento_medio_responsavel_sm_estimado_norm`;
   - `pct_alfabetizacao_15mais_estimado_norm`;
   - `pct_sem_banheiro_sanitario_estimado_norm`;
   - `pct_esgotamento_precario_estimado_norm`;
   - `pct_sem_rede_geral_agua_estimado_norm`;
   - `pct_lixo_destino_inadequado_estimado_norm`;
   - `IQV_final`.

In [7]:
cols_int = [
    "FLAG_EQUIPAMENTO_URBANO",
    "FLAG_FLAGRANTE",
    "OCORRENCIA_MES",
    "OCORRENCIA_DIA",
    "MADRUGADA",
    "MANHA",
    "TARDE",
    "NOITE",
    "CRIME_VIOLENTO",
    "ATENDIMENTO_OPERACIONAL_ASSISTENCIAL",
    "ACIDENTE_TRANSITO",
    "CRIME_PATRIMONIAL",
    "CRIME_DROGAS_SUBSTANCIAS",
    "CRIME_ORDEM_PUBLICA",
    "OUTROS",
    "ano",
    "populacao_estimado"
]

In [8]:
for col in cols_int:
    df_ml[col] = pd.to_numeric(df_ml[col], errors="coerce").astype("int64")

In [9]:
cols_float = [
    "log_rendimento",
    "log_pop",
    "rendimento_medio_responsavel_sm_estimado_norm",
    "pct_alfabetizacao_15mais_estimado_norm",
    "pct_sem_banheiro_sanitario_estimado_norm",
    "pct_esgotamento_precario_estimado_norm",
    "pct_sem_rede_geral_agua_estimado_norm",
    "pct_lixo_destino_inadequado_estimado_norm",
    "IQV_final",
]

In [10]:
for col in cols_float:
    df_ml[col] = pd.to_numeric(df_ml[col], errors="coerce").astype("float64")

### Definição da variável-alvo 

Conforme definido no projeto, o objetivo é desenvolver um modelo de classificação multiclasse para prever o tipo de ocorrência registrado no dataset. Nesse contexto, cada classe representa uma categoria de ocorrência. 

Como algumas categorias possuem baixa representatividade, elas foram agrupadas em uma única classe denominada `OUTROS`. Essa estratégia reduz a fragmentação das classes e constribui para tornar o problema de classificação mais estável. As classes consideradas na modelagem são:

- `CRIME_VIOLENTO`;
- `ATENDIMENTO_OPERACIONAL_ASSISTENCIAL`;
- `ACIDENTE_TRANSITO`;
- `CRIME_PATRIMONIAL`;
- `CRIME_DROGAS_SUBSTANCIAS`;
- `CRIME_ORDEM_PUBLICA`;
- `OUTROS`.

Além disso, considerando que a distribuição das classes é desbalanceada, será utilizada uma separação estratificada entre treino/validação e teste. Essa abordagem permite preservar, em cada subconjunto, proporções semelhantes às observadas na base original, tornando a validação mais consistente.

In [11]:
target_cols = [ 
    "CRIME_VIOLENTO",
    "ATENDIMENTO_OPERACIONAL_ASSISTENCIAL",
    "ACIDENTE_TRANSITO",
    "CRIME_PATRIMONIAL",
    "CRIME_DROGAS_SUBSTANCIAS",
    "CRIME_ORDEM_PUBLICA",
    "OUTROS"
]

In [12]:
df_ml[target_cols] = (
    df_ml[target_cols]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
    .astype(int)
)

In [13]:
df_ml["n_classes"] = df_ml[target_cols].sum(axis=1)

df_ml["n_classes"].value_counts().sort_index()

n_classes
1    582773
Name: count, dtype: int64

In [14]:
df_ml["categoria"] = df_ml[target_cols].idxmax(axis=1)

df_ml["categoria"].value_counts(normalize=True)

categoria
ATENDIMENTO_OPERACIONAL_ASSISTENCIAL    0.409415
ACIDENTE_TRANSITO                       0.183852
CRIME_PATRIMONIAL                       0.122809
CRIME_VIOLENTO                          0.095950
OUTROS                                  0.079498
CRIME_ORDEM_PUBLICA                     0.067886
CRIME_DROGAS_SUBSTANCIAS                0.040590
Name: proportion, dtype: float64

### 1. Primeira abordagem: indicadores separados

Nessa primeira abordagem, foram selecionadas as variáveis explicativas que serão utilizadas para prever a categoria do atendimento. O modelo considera informações espaciais, temporais, flags operacionais e indicadores socioeconômicos associados aos bairros. Nesta etapa, a ideia é utilizar separadamente as variáveis que compõe o Índice de Qualidade de Vida (IQV) calculado. 

A ideia é avaliar o efeito individual de cada indicador socioeconômico sobre a classificação das ocorrências, evitando que essas informações sejam resumidas previamente em um único indicador agregado. 

As variáveis socioeconômicas consideradas nesta abordagem são:

- `pct_alfabetizacao_15mais_estimado_norm`;
- `pct_sem_banheiro_sanitario_estimado_norm`;
- `pct_esgotamento_precario_estimado_norm`;
- `pct_sem_rede_geral_agua_estimado_norm`;
- `pct_lixo_destino_inadequado_estimado_norm`

In [15]:
features_abordagem_1 = [
    "ATENDIMENTO_BAIRRO_NOME",
    "FLAG_EQUIPAMENTO_URBANO",
    "FLAG_FLAGRANTE",
    "OCORRENCIA_DIA_SEMANA",
    "OCORRENCIA_MES",
    "MADRUGADA",
    "MANHA",
    "TARDE",
    "NOITE",
    "log_rendimento",
    "log_pop",
    "pct_alfabetizacao_15mais_estimado_norm",
    "pct_sem_banheiro_sanitario_estimado_norm",
    "pct_esgotamento_precario_estimado_norm",
    "pct_sem_rede_geral_agua_estimado_norm",
    "pct_lixo_destino_inadequado_estimado_norm",
]

### Codificação da variável-alvo 

Após a definição das classes finais, a variáel-alvo foi organizada em uma única coluna chamda `categoria`. Essa coluna representa a classe de ocorrência associada a cada registro da base. 

AS categoriais textuais foram convertidas em números utilizando `LabelEncoder`. Dessa forma, cada classe recebe um código numérico, mantendo a correspondência entre o nome original da categoria e o valor codificado.

In [16]:
X = df_ml[features_abordagem_1].copy()
y = df_ml["categoria"].copy()

In [17]:
le = LabelEncoder()

y_encoded = le.fit_transform(y)

le.classes_

array(['ACIDENTE_TRANSITO', 'ATENDIMENTO_OPERACIONAL_ASSISTENCIAL',
       'CRIME_DROGAS_SUBSTANCIAS', 'CRIME_ORDEM_PUBLICA',
       'CRIME_PATRIMONIAL', 'CRIME_VIOLENTO', 'OUTROS'], dtype=object)

### Separar treino, validação e teste

In [18]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y_encoded,
    test_size=0.40,
    stratify=y_encoded,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

In [19]:
print("Treino:", X_train.shape)
print("Validação:", X_val.shape)
print("Teste:", X_test.shape)

Treino: (349663, 16)
Validação: (116555, 16)
Teste: (116555, 16)


### Conferindo a distribuição das classes

Agora, iremos verificar se a separação estratificada manteve a proporção das classes durante a separação do dataset.

In [20]:
distribuicao_classes = pd.DataFrame({
    "classe": le.classes_,
    "treino": pd.Series(y_train).value_counts(normalize=True).sort_index().values,
    "validacao": pd.Series(y_val).value_counts(normalize=True).sort_index().values,
    "teste": pd.Series(y_test).value_counts(normalize=True).sort_index().values,
})

distribuicao_classes

,classe,treino,validacao,teste
0,ACIDENTE_TRANSITO,0.183851,0.183853,0.183853
1,ATENDIMENTO_OPERACIONAL_ASSISTENCIAL,0.409414,0.409412,0.409420
2,CRIME_DROGAS_SUBSTANCIAS,0.040591,0.040590,0.040590
3,CRIME_ORDEM_PUBLICA,0.067885,0.067891,0.067882
4,CRIME_PATRIMONIAL,0.122810,0.122809,0.122809
5,CRIME_VIOLENTO,0.095950,0.095946,0.095955
6,OUTROS,0.079499,0.079499,0.079490


### Criar preprocessador dinâmico

In [21]:
variavel_categoria = ["ATENDIMENTO_BAIRRO_NOME"]

In [22]:
def criar_onehot():
    return OneHotEncoder(handle_unknown="ignore", sparse_output=True)

In [23]:
def criar_preprocessador(features):
    categoricas = [col for col in variavel_categoria if col in features]
    numericas = [col for col in features if col not in categoricas]

    preprocessador = ColumnTransformer(
        transformers=[
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", criar_onehot())
            ]), categoricas),

            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), numericas)
        ],
        remainder="drop"
    )

    return preprocessador

### Função de treino para um subconjunto de features

In [24]:
def treinar_avaliar_subconjunto(
    nome_modelo,
    modelo,
    features,
    X_train,
    y_train,
    X_val,
    y_val,
    le,
    usar_sample_weight=False,
    imprimir_relatorio=False
):
    preprocessador = criar_preprocessador(features)

    pipe = Pipeline([
        ("preprocessador", preprocessador),
        ("modelo", clone(modelo))
    ])

    if usar_sample_weight:
        sample_weight = compute_sample_weight(
            class_weight="balanced",
            y=y_train
        )

        pipe.fit(
            X_train[features],
            y_train,
            modelo__sample_weight=sample_weight
        )
    else:
        pipe.fit(X_train[features], y_train)

    y_pred = pipe.predict(X_val[features])

    resultado = {
        "modelo": nome_modelo,
        "n_features": len(features),
        "features": features.copy(),
        "accuracy": accuracy_score(y_val, y_pred),
        "f1_macro": f1_score(y_val, y_pred, average="macro"),
        "f1_weighted": f1_score(y_val, y_pred, average="weighted")
    }

    if imprimir_relatorio:
        print(f"\nModelo: {nome_modelo}")
        print(f"Nº features: {len(features)}")
        print(f"Accuracy: {resultado['accuracy']:.4f}")
        print(f"F1 macro: {resultado['f1_macro']:.4f}")
        print(f"F1 weighted: {resultado['f1_weighted']:.4f}")

        print("\nRelatório por classe:")
        print(classification_report(
            y_val,
            y_pred,
            target_names=le.classes_,
            zero_division=0
        ))

    return resultado, pipe

### Função de redução progressiva

In [25]:
def reducao_progressiva_features(
    nome_modelo,
    modelo,
    X_train,
    y_train,
    X_val,
    y_val,
    features_iniciais,
    le,
    usar_sample_weight=False,
    metrica_escolha="f1_macro",
    n_repeats_importancia=5,
    random_state=42,
    pasta_saida="resultados_reducao_features"
):
    pasta_saida = Path(pasta_saida)
    pasta_saida.mkdir(exist_ok=True)

    print("=" * 80)
    print(f"Iniciando modelo: {nome_modelo}")
    print("=" * 80)

    resultados = []

    resultado_full, pipe_full = treinar_avaliar_subconjunto(
        nome_modelo=nome_modelo,
        modelo=modelo,
        features=features_iniciais,
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        le=le,
        usar_sample_weight=usar_sample_weight,
        imprimir_relatorio=True
    )

    resultado_full["features_removidas"] = []
    resultados.append(resultado_full)

    melhor_resultado = resultado_full.copy()
    melhor_pipe_path = pasta_saida / f"melhor_modelo_{nome_modelo}.joblib"

    print("\nCalculando importância das variáveis...")

    importancia = permutation_importance(
        pipe_full,
        X_val[features_iniciais],
        y_val,
        scoring=metrica_escolha,
        n_repeats=n_repeats_importancia,
        random_state=random_state,
        n_jobs=1,
        max_samples=min(20000, len(X_val))
    )

    df_importancia = pd.DataFrame({
        "feature": features_iniciais,
        "importancia_media": importancia.importances_mean,
        "importancia_desvio": importancia.importances_std
    }).sort_values("importancia_media", ascending=True)

    caminho_importancia = pasta_saida / f"importancia_{nome_modelo}.csv"
    df_importancia.to_csv(caminho_importancia, index=False)

    ordem_remocao = df_importancia["feature"].tolist()

    del pipe_full
    gc.collect()

    features_atuais = features_iniciais.copy()
    features_removidas = []
    resultado_atual = resultado_full.copy()

    for feature_remover in ordem_remocao:

        if len(features_atuais) <= 1:
            break

        features_candidatas = [
            f for f in features_atuais
            if f != feature_remover
        ]
        print("\n" + "-" * 80)
        print(f"Testando remoção da variável: {feature_remover}")
        print(f"Nº de features candidatas: {len(features_candidatas)}")

        resultado, pipe = treinar_avaliar_subconjunto(
            nome_modelo=nome_modelo,
            modelo=modelo,
            features=features_candidatas,
            X_train=X_train,
            y_train=y_train,
            X_val=X_val,
            y_val=y_val,
            le=le,
            usar_sample_weight=usar_sample_weight,
            imprimir_relatorio=False
        )

        delta = resultado[metrica_escolha] - resultado_atual[metrica_escolha]

        if delta >= 0:
            features_atuais = features_candidatas
            features_removidas.append(feature_remover)
            decisao = "removeu"
        else:
            decisao = "manteve"

        resultado["feature_testada"] = feature_remover
        resultado["decisao"] = decisao 
        resultado["delta_metrica"] = delta
        resultado["features_removidas"] = features_removidas.copy()

        if decisao == "removeu":
            resultado_atual = resultado.copy()

            melhorou = resultado[metrica_escolha] > melhor_resultado[metrica_escolha]
            empatou_com_menos_features = (
                resultado[metrica_escolha] == melhor_resultado[metrica_escolha]
                and resultado["n_features"] < melhor_resultado["n_features"]
            )

            if melhorou or empatou_com_menos_features:
                melhor_resultado = resultado.copy()

                print(f"Novo melhor modelo encontrado: {metrica_escolha} = {resultado[metrica_escolha]:.4f}")

        resultados.append(resultado)

        print(f"Accuracy: {resultado['accuracy']:.4f}")
        print(f"F1 macro: {resultado['f1_macro']:.4f}")
        print(f"F1 weighted: {resultado['f1_weighted']:.4f}")

        del pipe
        gc.collect()

    df_resultados = pd.DataFrame(resultados)

    caminho_resultados = pasta_saida / f"resultados_{nome_modelo}.csv"
    df_resultados.to_csv(caminho_resultados, index=False)

    print("\nTreinando e salvando o melhor modelo final...")

    _, melhor_pipe = treinar_avaliar_subconjunto(
        nome_modelo=nome_modelo,
        modelo=modelo,
        features=melhor_resultado["features"],
        X_train=X_train,
        y_train=y_train,
        X_val=X_val,
        y_val=y_val,
        le=le,
        usar_sample_weight=usar_sample_weight,
        imprimir_relatorio=False
    )

    joblib.dump(melhor_pipe, melhor_pipe_path, compress=3)
    del melhor_pipe
    gc.collect()

    resumo = {
        "modelo": nome_modelo,
        "metrica_escolha": metrica_escolha,
        "melhor_accuracy": melhor_resultado["accuracy"],
        "melhor_f1_macro": melhor_resultado["f1_macro"],
        "melhor_f1_weighted": melhor_resultado["f1_weighted"],
        "melhor_n_features": melhor_resultado["n_features"],
        "melhores_features": melhor_resultado["features"],
        "features_removidas": melhor_resultado["features_removidas"],
        "caminho_melhor_modelo": str(melhor_pipe_path)
    }

    caminho_resumo = pasta_saida / f"resumo_{nome_modelo}.json"

    with open(caminho_resumo, "w", encoding="utf-8") as f:
        json.dump(resumo, f, ensure_ascii=False, indent=4)

    print("\n" + "=" * 80)
    print(f"Finalizado: {nome_modelo}")
    print(f"Melhor {metrica_escolha}: {melhor_resultado[metrica_escolha]:.4f}")
    print(f"Nº de features no melhor modelo: {melhor_resultado['n_features']}")
    print(f"Melhor modelo salvo em: {melhor_pipe_path}")
    print("=" * 80)

    return df_resultados, df_importancia, resumo

### Criar lista para guardar os resumos

In [26]:
todos_resumos = []

### Regressão logística 

Colocar bla bla bla

In [27]:
modelo_logistico = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

resultados_logistico, importancia_logistico, resumo_logistico = reducao_progressiva_features(
    nome_modelo="regressao_logistica",
    modelo=modelo_logistico,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    features_iniciais=features_abordagem_1,
    le=le,
    usar_sample_weight=False,
    metrica_escolha="f1_macro",
    pasta_saida="resultados_reducao_features"
)

Iniciando modelo: regressao_logistica

Modelo: regressao_logistica
Nº features: 16
Accuracy: 0.3829
F1 macro: 0.3399
F1 weighted: 0.3713

Relatório por classe:
                                      precision    recall  f1-score   support

                   ACIDENTE_TRANSITO       0.48      0.80      0.60     21429
ATENDIMENTO_OPERACIONAL_ASSISTENCIAL       0.69      0.23      0.35     47719
            CRIME_DROGAS_SUBSTANCIAS       0.16      0.45      0.24      4731
                 CRIME_ORDEM_PUBLICA       0.27      0.43      0.33      7913
                   CRIME_PATRIMONIAL       0.31      0.34      0.32     14314
                      CRIME_VIOLENTO       0.26      0.20      0.23     11183
                              OUTROS       0.26      0.40      0.31      9266

                            accuracy                           0.38    116555
                           macro avg       0.35      0.41      0.34    116555
                        weighted avg       0.48      0.38 

In [28]:
todos_resumos.append(resumo_logistico)

In [29]:
resultados_logistico.sort_values("f1_macro", ascending=False).head(10)

,modelo,n_features,features,accuracy,f1_macro,f1_weighted,features_removidas,feature_testada,decisao,delta_metrica
1,regressao_logistica,15,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.383776,0.340554,0.372520,[pct_lixo_destino_inadequado_estimado_norm],pct_lixo_destino_inadequado_estimado_norm,removeu,0.000672
8,regressao_logistica,14,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.383596,0.340393,0.372252,[pct_lixo_destino_inadequado_estimado_norm],NOITE,manteve,-0.000160
3,regressao_logistica,14,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.383278,0.340138,0.371721,[pct_lixo_destino_inadequado_estimado_norm],OCORRENCIA_DIA_SEMANA,manteve,-0.000415
11,regressao_logistica,14,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.383012,0.339937,0.371481,[pct_lixo_destino_inadequado_estimado_norm],MADRUGADA,manteve,-0.000617
0,regressao_logistica,16,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.382866,0.339882,0.371295,[],NaN,NaN,NaN
4,regressao_logistica,14,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.382901,0.339861,0.371368,[pct_lixo_destino_inadequado_estimado_norm],MANHA,manteve,-0.000693
2,regressao_logistica,14,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.382781,0.339823,0.371008,[pct_lixo_destino_inadequado_estimado_norm],OCORRENCIA_MES,manteve,-0.000730
5,regressao_logistica,14,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.382755,0.339739,0.371170,[pct_lixo_destino_inadequado_estimado_norm],TARDE,manteve,-0.000814
12,regressao_logistica,14,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.382463,0.339731,0.370765,[pct_lixo_destino_inadequado_estimado_norm],log_pop,manteve,-0.000823
6,regressao_logistica,14,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.382746,0.339597,0.371449,[pct_lixo_destino_inadequado_estimado_norm],pct_esgotamento_precario_estimado_norm,manteve,-0.000957


In [30]:
importancia_logistico.sort_values("importancia_media", ascending=False)

,feature,importancia_media,importancia_desvio
0,ATENDIMENTO_BAIRRO_NOME,0.144646,0.001979
9,log_rendimento,0.125112,0.003137
11,pct_alfabetizacao_15mais_estimado_norm,0.115106,0.002019
1,FLAG_EQUIPAMENTO_URBANO,0.074901,0.001034
10,log_pop,0.036157,0.001450
5,MADRUGADA,0.024855,0.000572
2,FLAG_FLAGRANTE,0.011305,0.000847
12,pct_sem_banheiro_sanitario_estimado_norm,0.006217,0.001042
8,NOITE,0.005827,0.001035
14,pct_sem_rede_geral_agua_estimado_norm,0.002441,0.000499


In [31]:
del modelo_logistico
del resultados_logistico
del importancia_logistico
gc.collect()

33

### Árvore de decisão 

Colocar bla bla bla

In [32]:
modelo_arvore = DecisionTreeClassifier(
    class_weight="balanced",
    random_state=42
)

resultados_arvore, importancia_arvore, resumo_arvore = reducao_progressiva_features(
    nome_modelo="arvore_decisao",
    modelo=modelo_arvore,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    features_iniciais=features_abordagem_1,
    le=le,
    usar_sample_weight=False,
    metrica_escolha="f1_macro",
    pasta_saida="resultados_reducao_features"
)

Iniciando modelo: arvore_decisao

Modelo: arvore_decisao
Nº features: 16
Accuracy: 0.3873
F1 macro: 0.3259
F1 weighted: 0.3995

Relatório por classe:
                                      precision    recall  f1-score   support

                   ACIDENTE_TRANSITO       0.54      0.56      0.55     21429
ATENDIMENTO_OPERACIONAL_ASSISTENCIAL       0.57      0.41      0.48     47719
            CRIME_DROGAS_SUBSTANCIAS       0.14      0.28      0.19      4731
                 CRIME_ORDEM_PUBLICA       0.26      0.37      0.30      7913
                   CRIME_PATRIMONIAL       0.26      0.27      0.27     14314
                      CRIME_VIOLENTO       0.20      0.24      0.22     11183
                              OUTROS       0.25      0.33      0.28      9266

                            accuracy                           0.39    116555
                           macro avg       0.32      0.35      0.33    116555
                        weighted avg       0.43      0.39      0.40 

In [33]:
todos_resumos.append(resumo_arvore)

In [34]:
resultados_arvore.sort_values("f1_macro", ascending=False).head(10)

,modelo,n_features,features,accuracy,f1_macro,f1_weighted,features_removidas,feature_testada,decisao,delta_metrica
13,arvore_decisao,10,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.428133,0.382438,0.433037,"[TARDE, MANHA, pct_lixo_destino_inadequado_est...",pct_sem_banheiro_sanitario_estimado_norm,removeu,0.000026
8,arvore_decisao,11,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.428124,0.382412,0.433037,"[TARDE, MANHA, pct_lixo_destino_inadequado_est...",OCORRENCIA_MES,removeu,0.028235
14,arvore_decisao,9,"[FLAG_EQUIPAMENTO_URBANO, FLAG_FLAGRANTE, MADR...",0.428090,0.382406,0.432996,"[TARDE, MANHA, pct_lixo_destino_inadequado_est...",ATENDIMENTO_BAIRRO_NOME,manteve,-0.000032
10,arvore_decisao,10,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.428064,0.382350,0.432971,"[TARDE, MANHA, pct_lixo_destino_inadequado_est...",pct_alfabetizacao_15mais_estimado_norm,manteve,-0.000062
16,arvore_decisao,9,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.428013,0.382324,0.432920,"[TARDE, MANHA, pct_lixo_destino_inadequado_est...",pct_esgotamento_precario_estimado_norm,manteve,-0.000114
9,arvore_decisao,10,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.428021,0.382302,0.432928,"[TARDE, MANHA, pct_lixo_destino_inadequado_est...",pct_sem_rede_geral_agua_estimado_norm,manteve,-0.000110
12,arvore_decisao,10,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.428038,0.382290,0.432937,"[TARDE, MANHA, pct_lixo_destino_inadequado_est...",log_rendimento,manteve,-0.000123
11,arvore_decisao,10,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.427935,0.382240,0.432893,"[TARDE, MANHA, pct_lixo_destino_inadequado_est...",log_pop,manteve,-0.000172
6,arvore_decisao,12,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.406280,0.354178,0.417168,"[TARDE, MANHA, pct_lixo_destino_inadequado_est...",OCORRENCIA_DIA_SEMANA,removeu,0.026169
15,arvore_decisao,9,"[ATENDIMENTO_BAIRRO_NOME, FLAG_FLAGRANTE, MADR...",0.404787,0.353723,0.410332,"[TARDE, MANHA, pct_lixo_destino_inadequado_est...",FLAG_EQUIPAMENTO_URBANO,manteve,-0.028715


In [35]:
importancia_arvore.sort_values("importancia_media", ascending=False)

,feature,importancia_media,importancia_desvio
13,pct_esgotamento_precario_estimado_norm,0.054559,0.001465
1,FLAG_EQUIPAMENTO_URBANO,0.054129,0.001971
0,ATENDIMENTO_BAIRRO_NOME,0.053844,0.001099
12,pct_sem_banheiro_sanitario_estimado_norm,0.053586,0.001238
9,log_rendimento,0.052722,0.001522
10,log_pop,0.049857,0.001311
11,pct_alfabetizacao_15mais_estimado_norm,0.041224,0.002021
14,pct_sem_rede_geral_agua_estimado_norm,0.039694,0.001456
4,OCORRENCIA_MES,0.031461,0.002628
5,MADRUGADA,0.030152,0.001892


In [36]:
del modelo_arvore
del resultados_arvore
del importancia_arvore
gc.collect()

33

### Random forest

Colocar bla bla bla 

In [37]:
modelo_rf = RandomForestClassifier(
    n_estimators=150,
    max_depth=20,
    min_samples_leaf=5,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

resultados_rf, importancia_rf, resumo_rf = reducao_progressiva_features(
    nome_modelo="random_forest",
    modelo=modelo_rf,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    features_iniciais=features_abordagem_1,
    le=le,
    usar_sample_weight=False,
    metrica_escolha="f1_macro",
    pasta_saida="resultados_reducao_features"
)

Iniciando modelo: random_forest

Modelo: random_forest
Nº features: 16
Accuracy: 0.4541
F1 macro: 0.4052
F1 weighted: 0.4582

Relatório por classe:
                                      precision    recall  f1-score   support

                   ACIDENTE_TRANSITO       0.54      0.77      0.64     21429
ATENDIMENTO_OPERACIONAL_ASSISTENCIAL       0.71      0.37      0.49     47719
            CRIME_DROGAS_SUBSTANCIAS       0.20      0.44      0.27      4731
                 CRIME_ORDEM_PUBLICA       0.38      0.46      0.42      7913
                   CRIME_PATRIMONIAL       0.35      0.36      0.36     14314
                      CRIME_VIOLENTO       0.28      0.33      0.31     11183
                              OUTROS       0.30      0.43      0.36      9266

                            accuracy                           0.45    116555
                           macro avg       0.40      0.45      0.41    116555
                        weighted avg       0.52      0.45      0.46   

In [38]:
todos_resumos.append(resumo_rf)

In [39]:
resultados_rf.sort_values("f1_macro", ascending=False).head(10)

,modelo,n_features,features,accuracy,f1_macro,f1_weighted,features_removidas,feature_testada,decisao,delta_metrica
15,random_forest,14,"[FLAG_EQUIPAMENTO_URBANO, FLAG_FLAGRANTE, OCOR...",0.459869,0.407985,0.466598,"[MANHA, ATENDIMENTO_BAIRRO_NOME]",ATENDIMENTO_BAIRRO_NOME,removeu,0.000985
1,random_forest,15,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.456102,0.407000,0.459760,[MANHA],MANHA,removeu,0.001811
2,random_forest,14,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.455442,0.406082,0.458823,[MANHA],pct_lixo_destino_inadequado_estimado_norm,manteve,-0.000918
5,random_forest,14,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.454738,0.406008,0.457773,[MANHA],log_pop,manteve,-0.000992
7,random_forest,14,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.454704,0.405465,0.458103,[MANHA],pct_sem_rede_geral_agua_estimado_norm,manteve,-0.001535
9,random_forest,14,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.454240,0.405293,0.457352,[MANHA],pct_esgotamento_precario_estimado_norm,manteve,-0.001707
0,random_forest,16,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.454138,0.405188,0.458176,[],NaN,NaN,NaN
12,random_forest,14,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.453666,0.405043,0.456891,[MANHA],pct_alfabetizacao_15mais_estimado_norm,manteve,-0.001956
3,random_forest,14,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.454850,0.404731,0.458598,[MANHA],TARDE,manteve,-0.002269
6,random_forest,14,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.453468,0.404634,0.456325,[MANHA],log_rendimento,manteve,-0.002366


In [40]:
importancia_rf.sort_values("importancia_media", ascending=False)

,feature,importancia_media,importancia_desvio
1,FLAG_EQUIPAMENTO_URBANO,0.084368,0.002462
0,ATENDIMENTO_BAIRRO_NOME,0.029305,0.001194
5,MADRUGADA,0.025395,0.001235
2,FLAG_FLAGRANTE,0.018071,0.001029
11,pct_alfabetizacao_15mais_estimado_norm,0.017098,0.001571
12,pct_sem_banheiro_sanitario_estimado_norm,0.016825,0.000711
4,OCORRENCIA_MES,0.015334,0.001481
13,pct_esgotamento_precario_estimado_norm,0.013808,0.000879
3,OCORRENCIA_DIA_SEMANA,0.012042,0.001355
14,pct_sem_rede_geral_agua_estimado_norm,0.007738,0.000589


In [41]:
del modelo_rf
del resultados_rf
del importancia_rf
gc.collect()

33

### XGBoost

Colocar bla bla bla

In [42]:
modelo_xgb = XGBClassifier(
    objective="multi:softprob",
    num_class=len(le.classes_),
    eval_metric="mlogloss",
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

resultados_xgb, importancia_xgb, resumo_xgb = reducao_progressiva_features(
    nome_modelo="xgboost",
    modelo=modelo_xgb,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    features_iniciais=features_abordagem_1,
    le=le,
    usar_sample_weight=True,
    metrica_escolha="f1_macro",
    pasta_saida="resultados_reducao_features"
)

Iniciando modelo: xgboost

Modelo: xgboost
Nº features: 16
Accuracy: 0.4392
F1 macro: 0.3935
F1 weighted: 0.4409

Relatório por classe:
                                      precision    recall  f1-score   support

                   ACIDENTE_TRANSITO       0.53      0.78      0.63     21429
ATENDIMENTO_OPERACIONAL_ASSISTENCIAL       0.72      0.34      0.46     47719
            CRIME_DROGAS_SUBSTANCIAS       0.19      0.47      0.28      4731
                 CRIME_ORDEM_PUBLICA       0.36      0.45      0.40      7913
                   CRIME_PATRIMONIAL       0.35      0.36      0.35     14314
                      CRIME_VIOLENTO       0.29      0.31      0.30     11183
                              OUTROS       0.28      0.43      0.34      9266

                            accuracy                           0.44    116555
                           macro avg       0.39      0.45      0.39    116555
                        weighted avg       0.52      0.44      0.44    116555


Ca

In [43]:
todos_resumos.append(resumo_xgb)

In [44]:
resultados_xgb.sort_values("f1_macro", ascending=False).head(10)

,modelo,n_features,features,accuracy,f1_macro,f1_weighted,features_removidas,feature_testada,decisao,delta_metrica
15,xgboost,14,"[FLAG_EQUIPAMENTO_URBANO, FLAG_FLAGRANTE, OCOR...",0.443130,0.396843,0.445771,"[NOITE, ATENDIMENTO_BAIRRO_NOME]",ATENDIMENTO_BAIRRO_NOME,removeu,0.002396
6,xgboost,15,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.440462,0.394447,0.442270,[NOITE],NOITE,removeu,0.000904
13,xgboost,14,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.439458,0.393653,0.442290,[NOITE],pct_sem_banheiro_sanitario_estimado_norm,manteve,-0.000794
0,xgboost,16,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.439192,0.393542,0.440863,[],NaN,NaN,NaN
4,xgboost,15,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.439629,0.393450,0.441531,[],TARDE,manteve,-0.000092
1,xgboost,15,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.439295,0.393437,0.441114,[],MANHA,manteve,-0.000106
8,xgboost,14,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.438892,0.393250,0.440582,[NOITE],log_pop,manteve,-0.001197
5,xgboost,15,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.438608,0.393231,0.440132,[],pct_alfabetizacao_15mais_estimado_norm,manteve,-0.000312
11,xgboost,14,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.439544,0.393175,0.441292,[NOITE],log_rendimento,manteve,-0.001272
12,xgboost,14,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.438488,0.392836,0.440508,[NOITE],pct_sem_rede_geral_agua_estimado_norm,manteve,-0.001611


In [45]:
importancia_xgb.sort_values("importancia_media", ascending=False)

,feature,importancia_media,importancia_desvio
1,FLAG_EQUIPAMENTO_URBANO,0.084798,0.001844
0,ATENDIMENTO_BAIRRO_NOME,0.052696,0.000379
5,MADRUGADA,0.037297,0.000999
12,pct_sem_banheiro_sanitario_estimado_norm,0.036436,0.000689
14,pct_sem_rede_geral_agua_estimado_norm,0.026468,0.001348
9,log_rendimento,0.025237,0.000579
13,pct_esgotamento_precario_estimado_norm,0.021995,0.001369
2,FLAG_FLAGRANTE,0.021660,0.000575
10,log_pop,0.017392,0.001120
3,OCORRENCIA_DIA_SEMANA,0.013114,0.001099


In [46]:
del modelo_xgb
del resultados_xgb
del importancia_xgb
gc.collect()

33

### CatBoost 

Colocar bla bla bla

In [47]:
modelo_catboost = CatBoostClassifier(
    loss_function="MultiClass",
    iterations=300,
    learning_rate=0.05,
    depth=6,
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=False
)

resultados_catboost, importancia_catboost, resumo_catboost = reducao_progressiva_features(
    nome_modelo="catboost",
    modelo=modelo_catboost,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    features_iniciais=features_abordagem_1,
    le=le,
    usar_sample_weight=False,
    metrica_escolha="f1_macro",
    pasta_saida="resultados_reducao_features"
)

Iniciando modelo: catboost

Modelo: catboost
Nº features: 16
Accuracy: 0.4174
F1 macro: 0.3737
F1 weighted: 0.4153

Relatório por classe:
                                      precision    recall  f1-score   support

                   ACIDENTE_TRANSITO       0.50      0.78      0.61     21429
ATENDIMENTO_OPERACIONAL_ASSISTENCIAL       0.71      0.30      0.42     47719
            CRIME_DROGAS_SUBSTANCIAS       0.18      0.47      0.26      4731
                 CRIME_ORDEM_PUBLICA       0.33      0.45      0.38      7913
                   CRIME_PATRIMONIAL       0.34      0.36      0.35     14314
                      CRIME_VIOLENTO       0.30      0.26      0.27     11183
                              OUTROS       0.26      0.43      0.32      9266

                            accuracy                           0.42    116555
                           macro avg       0.37      0.43      0.37    116555
                        weighted avg       0.50      0.42      0.42    116555




In [48]:
todos_resumos.append(resumo_catboost)

In [49]:
resultados_catboost.sort_values("f1_macro", ascending=False).head(10)

,modelo,n_features,features,accuracy,f1_macro,f1_weighted,features_removidas,feature_testada,decisao,delta_metrica
12,catboost,15,"[FLAG_EQUIPAMENTO_URBANO, FLAG_FLAGRANTE, OCOR...",0.420428,0.375767,0.420035,[ATENDIMENTO_BAIRRO_NOME],ATENDIMENTO_BAIRRO_NOME,removeu,0.002111
0,catboost,16,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.417374,0.373657,0.415337,[],NaN,NaN,NaN
14,catboost,14,"[FLAG_EQUIPAMENTO_URBANO, FLAG_FLAGRANTE, OCOR...",0.417288,0.373147,0.415080,[ATENDIMENTO_BAIRRO_NOME],MADRUGADA,manteve,-0.002620
6,catboost,15,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.416662,0.372713,0.414518,[],pct_lixo_destino_inadequado_estimado_norm,manteve,-0.000944
4,catboost,15,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.416216,0.372652,0.413727,[],NOITE,manteve,-0.001004
13,catboost,14,"[FLAG_EQUIPAMENTO_URBANO, FLAG_FLAGRANTE, OCOR...",0.416113,0.372544,0.415214,[ATENDIMENTO_BAIRRO_NOME],pct_sem_rede_geral_agua_estimado_norm,manteve,-0.003223
3,catboost,15,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.415907,0.372489,0.413099,[],TARDE,manteve,-0.001168
8,catboost,15,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.415649,0.372058,0.413074,[],pct_alfabetizacao_15mais_estimado_norm,manteve,-0.001599
5,catboost,15,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.417442,0.371995,0.416088,[],OCORRENCIA_DIA_SEMANA,manteve,-0.001662
11,catboost,15,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",0.416078,0.371658,0.413626,[],pct_esgotamento_precario_estimado_norm,manteve,-0.001999


In [50]:
importancia_catboost.sort_values("importancia_media", ascending=False)

,feature,importancia_media,importancia_desvio
1,FLAG_EQUIPAMENTO_URBANO,0.087449,0.001508
12,pct_sem_banheiro_sanitario_estimado_norm,0.039970,0.001783
5,MADRUGADA,0.035344,0.001351
14,pct_sem_rede_geral_agua_estimado_norm,0.030978,0.002331
0,ATENDIMENTO_BAIRRO_NOME,0.026991,0.000616
13,pct_esgotamento_precario_estimado_norm,0.023268,0.000658
2,FLAG_FLAGRANTE,0.019442,0.000961
10,log_pop,0.015643,0.000321
11,pct_alfabetizacao_15mais_estimado_norm,0.015285,0.001513
9,log_rendimento,0.011316,0.000953


In [51]:
del modelo_catboost
del resultados_catboost
del importancia_catboost
gc.collect()

33

### Comparar os modelos rodados

In [52]:
comparacao_modelos = pd.DataFrame(todos_resumos)

comparacao_modelos = comparacao_modelos.sort_values(
    "melhor_f1_macro",
    ascending=False
)

comparacao_modelos

,modelo,metrica_escolha,melhor_accuracy,melhor_f1_macro,melhor_f1_weighted,melhor_n_features,melhores_features,features_removidas,caminho_melhor_modelo
2,random_forest,f1_macro,0.459869,0.407985,0.466598,14,"[FLAG_EQUIPAMENTO_URBANO, FLAG_FLAGRANTE, OCOR...","[MANHA, ATENDIMENTO_BAIRRO_NOME]",resultados_reducao_features/melhor_modelo_rand...
3,xgboost,f1_macro,0.443130,0.396843,0.445771,14,"[FLAG_EQUIPAMENTO_URBANO, FLAG_FLAGRANTE, OCOR...","[NOITE, ATENDIMENTO_BAIRRO_NOME]",resultados_reducao_features/melhor_modelo_xgbo...
1,arvore_decisao,f1_macro,0.428133,0.382438,0.433037,10,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...","[TARDE, MANHA, pct_lixo_destino_inadequado_est...",resultados_reducao_features/melhor_modelo_arvo...
4,catboost,f1_macro,0.420428,0.375767,0.420035,15,"[FLAG_EQUIPAMENTO_URBANO, FLAG_FLAGRANTE, OCOR...",[ATENDIMENTO_BAIRRO_NOME],resultados_reducao_features/melhor_modelo_catb...
0,regressao_logistica,f1_macro,0.383776,0.340554,0.372520,15,"[ATENDIMENTO_BAIRRO_NOME, FLAG_EQUIPAMENTO_URB...",[pct_lixo_destino_inadequado_estimado_norm],resultados_reducao_features/melhor_modelo_regr...


In [53]:
comparacao_modelos[[
    "modelo",
    "melhor_accuracy",
    "melhor_f1_macro",
    "melhor_f1_weighted",
    "melhor_n_features"
]]

,modelo,melhor_accuracy,melhor_f1_macro,melhor_f1_weighted,melhor_n_features
2,random_forest,0.459869,0.407985,0.466598,14
3,xgboost,0.443130,0.396843,0.445771,14
1,arvore_decisao,0.428133,0.382438,0.433037,10
4,catboost,0.420428,0.375767,0.420035,15
0,regressao_logistica,0.383776,0.340554,0.372520,15


In [54]:
melhor_linha = comparacao_modelos.iloc[0]

melhor_linha

modelo                                                       random_forest
metrica_escolha                                                   f1_macro
melhor_accuracy                                                   0.459869
melhor_f1_macro                                                   0.407985
melhor_f1_weighted                                                0.466598
melhor_n_features                                                       14
melhores_features        [FLAG_EQUIPAMENTO_URBANO, FLAG_FLAGRANTE, OCOR...
features_removidas                        [MANHA, ATENDIMENTO_BAIRRO_NOME]
caminho_melhor_modelo    resultados_reducao_features/melhor_modelo_rand...
Name: 2, dtype: object

In [55]:
melhor_modelo_path = melhor_linha["caminho_melhor_modelo"]
melhores_features = melhor_linha["melhores_features"]

melhor_pipe = joblib.load(melhor_modelo_path)

### Avaliação no dataset de teste

In [56]:
y_pred_test = melhor_pipe.predict(X_test[melhores_features])

In [57]:
print("Resultado final no teste:")
print("Accuracy:", accuracy_score(y_test, y_pred_test))
print("F1 macro:", f1_score(y_test, y_pred_test, average="macro"))
print("F1 weighted:", f1_score(y_test, y_pred_test, average="weighted"))

print("\nRelatório por classe:")
print(classification_report(
    y_test,
    y_pred_test,
    target_names=le.classes_,
    zero_division=0
))

Resultado final no teste:
Accuracy: 0.4573463171893098
F1 macro: 0.4037366551168172
F1 weighted: 0.4642591976602646

Relatório por classe:
                                      precision    recall  f1-score   support

                   ACIDENTE_TRANSITO       0.56      0.75      0.64     21429
ATENDIMENTO_OPERACIONAL_ASSISTENCIAL       0.70      0.39      0.51     47720
            CRIME_DROGAS_SUBSTANCIAS       0.19      0.43      0.26      4731
                 CRIME_ORDEM_PUBLICA       0.36      0.44      0.40      7912
                   CRIME_PATRIMONIAL       0.35      0.36      0.36     14314
                      CRIME_VIOLENTO       0.28      0.34      0.31     11184
                              OUTROS       0.31      0.41      0.35      9265

                            accuracy                           0.46    116555
                           macro avg       0.39      0.45      0.40    116555
                        weighted avg       0.52      0.46      0.46    116555

